# 🏠 RAGScore + Ollama — 100% Private Local RAG Evaluation

Run RAGScore entirely on your local machine with **Ollama** — no API keys, no cloud, no data leaves your machine.

| Feature | Details |
|---------|--------|
| 🔒 **Privacy** | All data stays on your machine |
| 💰 **Cost** | Completely free — no API charges |
| 🤖 **Models** | llama3.1, mistral, gemma2, qwen2.5, phi3, and more |
| ⚡ **Speed** | Fast with GPU, works on CPU too |

**Requirements:**
- [Ollama](https://ollama.ai) installed and running (`ollama serve`)
- A pulled model (e.g. `ollama pull llama3.1`)
- `ragscore >= 0.7.7`

> ⚠️ **Colab note:** Ollama requires a local machine. This notebook is designed to run
> on your **local Jupyter** environment, not on Colab (which has no Ollama).
> If you want to try it on Colab, you can install Ollama in the runtime (see Step 1b).

## 1a. Install RAGScore (local Jupyter)

In [ ]:
!pip install -q "ragscore[notebook]>=0.7.7" numpy

import nest_asyncio
nest_asyncio.apply()
print("✅ RAGScore installed")

## 1b. Install Ollama in Colab (optional — skip if running locally)

If you're on Colab, run the cell below to install and start Ollama in the runtime.
If you're on local Jupyter with Ollama already running, **skip this cell**.

In [ ]:
# Only run this on Colab!
import subprocess, time, os

# Install Ollama
!curl -fsSL https://ollama.ai/install.sh | sh

# Start Ollama server in background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)  # Wait for server to start

# Pull a small model (qwen2.5:1.5b is fast on Colab's free GPU)
!ollama pull qwen2.5:1.5b
print("✅ Ollama ready with qwen2.5:1.5b")

## 2. Verify Ollama is running

In [ ]:
import urllib.request, json

try:
    resp = urllib.request.urlopen("http://localhost:11434/api/tags")
    models = json.loads(resp.read())["models"]
    print("✅ Ollama is running!")
    print(f"\n📦 Available models ({len(models)}):")
    for m in models:
        size_gb = m.get("size", 0) / 1e9
        print(f"   • {m['name']} ({size_gb:.1f} GB)")
except Exception as e:
    print(f"❌ Ollama not running: {e}")
    print("\n💡 Start Ollama with: ollama serve")
    print("   Then pull a model: ollama pull llama3.1")

## 3. Choose your model

Set the model you want to use. Popular choices:

| Model | Size | Best for |
|-------|------|----------|
| `llama3.1` | 4.7 GB | Best overall quality |
| `llama3.1:70b` | 40 GB | Maximum quality (needs lots of RAM) |
| `mistral` | 4.1 GB | Fast, good quality |
| `gemma2` | 5.4 GB | Google's latest |
| `qwen2.5:1.5b` | 0.9 GB | Ultra-fast, great for testing |
| `phi3` | 2.3 GB | Small but capable |

In [ ]:
# Choose your model here:
OLLAMA_MODEL = "llama3.1"  # Change to any model you've pulled

print(f"🤖 Using model: {OLLAMA_MODEL}")

## 4. Create a sample document

In [ ]:
%%writefile sample_doc.txt
Retrieval-Augmented Generation (RAG) is an AI architecture that enhances large language models by combining them with external knowledge retrieval. Instead of relying solely on the model's training data, RAG systems first search a knowledge base for relevant documents, then use these documents as context when generating responses.

The RAG pipeline consists of three main stages: indexing, retrieval, and generation. During indexing, documents are split into chunks, converted to vector embeddings, and stored in a vector database. Popular vector databases include Pinecone, Weaviate, Qdrant, ChromaDB, and pgvector for PostgreSQL. The choice of embedding model significantly impacts retrieval quality — OpenAI's text-embedding-3-small and Cohere's embed-v3 are popular choices.

Retrieval is triggered when a user submits a query. The query is embedded using the same model, and the vector database returns the top-k most similar chunks based on cosine similarity or other distance metrics. Hybrid retrieval combines vector search with traditional keyword search (BM25) for better results. Re-ranking models like Cohere Rerank or cross-encoders can further improve relevance.

The generation stage feeds the retrieved context along with the original question to an LLM. The system prompt typically instructs the model to answer based only on the provided context, reducing hallucination. Common LLMs for RAG include GPT-4o, Claude 3.5, Llama 3.1, and Mistral. Temperature settings between 0.1 and 0.3 are recommended for factual QA.

Chunking strategy is critical for RAG performance. Fixed-size chunking (e.g., 512 tokens with 50-token overlap) is simple but may split sentences awkwardly. Semantic chunking groups related sentences together, while recursive character splitting offers a balance. Document-specific chunking can leverage headers and sections in structured documents.

Evaluating RAG systems requires measuring both retrieval quality and generation quality. Retrieval metrics include precision@k, recall@k, and NDCG. Generation metrics include faithfulness (does the answer match the source?), relevance (does it answer the question?), and completeness (does it cover all key points?). RAGScore automates this evaluation using LLM-as-judge methodology.

## 5. Build a mini RAG with Ollama

In [ ]:
import numpy as np
import urllib.request, json

# Read document
with open("sample_doc.txt") as f:
    text = f.read()
chunks = [p.strip() for p in text.split("\n\n") if len(p.strip()) > 50]
print(f"✅ {len(chunks)} chunks created")

def ollama_embed(text):
    """Get embeddings from Ollama."""
    data = json.dumps({"model": OLLAMA_MODEL, "input": text}).encode()
    req = urllib.request.Request(
        "http://localhost:11434/api/embed",
        data=data,
        headers={"Content-Type": "application/json"},
    )
    resp = json.loads(urllib.request.urlopen(req).read())
    return resp["embeddings"][0]

def ollama_chat(messages):
    """Chat with Ollama."""
    data = json.dumps({
        "model": OLLAMA_MODEL,
        "messages": messages,
        "stream": False,
        "options": {"temperature": 0.3},
    }).encode()
    req = urllib.request.Request(
        "http://localhost:11434/api/chat",
        data=data,
        headers={"Content-Type": "application/json"},
    )
    resp = json.loads(urllib.request.urlopen(req).read())
    return resp["message"]["content"]

# Embed all chunks
print("Embedding chunks...")
chunk_embeddings = np.array([ollama_embed(c) for c in chunks])
print(f"✅ {len(chunk_embeddings)} chunks embedded")

def my_rag(question):
    """Embed question → find top-3 chunks → generate answer with Ollama."""
    q_emb = np.array(ollama_embed(question))
    sims = chunk_embeddings @ q_emb / (
        np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(q_emb)
    )
    top_idx = np.argsort(sims)[-3:][::-1]
    context = "\n\n".join([chunks[i] for i in top_idx])

    return ollama_chat([
        {"role": "system", "content": "Answer based only on the provided context. Be concise and accurate."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ])

# Quick test
print("\n🧪 Test query:")
print(my_rag("What is RAG?"))

## 6. Evaluate RAG with Ollama (basic)

In [ ]:
from ragscore import quick_test

result = quick_test(
    my_rag,
    docs="sample_doc.txt",
    n=5,
    model=OLLAMA_MODEL,       # Use Ollama for QA generation
    judge_model=OLLAMA_MODEL, # Use Ollama for judging too
)

print(f"\n📊 Accuracy: {result.accuracy:.0%}")
print(f"📊 Avg Score: {result.avg_score:.1f}/5")

print("\n📋 Generated questions:")
for _, row in result.df.iterrows():
    print(f"  Q: {row['question'][:80]}")
    print(f"  Score: {row['score']}/5")
    print()

## 7. Detailed multi-metric evaluation

In [ ]:
detailed_result = quick_test(
    my_rag,
    docs="sample_doc.txt",
    n=5,
    model=OLLAMA_MODEL,
    judge_model=OLLAMA_MODEL,
    detailed=True,  # ⭐ 5-metric evaluation
)

# Show detailed metrics
display(detailed_result.df[[
    "question", "score", "correctness", "completeness",
    "relevance", "conciseness", "faithfulness"
]])

## 8. Audience-targeted evaluation

In [ ]:
# Developer-focused questions
dev_result = quick_test(
    my_rag,
    docs="sample_doc.txt",
    n=5,
    model=OLLAMA_MODEL,
    judge_model=OLLAMA_MODEL,
    audience="backend developers",
    purpose="building a RAG pipeline",
)

print("\n👨‍💻 Developer questions:")
for _, row in dev_result.df.iterrows():
    print(f"  Q: {row['question'][:80]}")

# Manager-focused questions
mgr_result = quick_test(
    my_rag,
    docs="sample_doc.txt",
    n=5,
    model=OLLAMA_MODEL,
    judge_model=OLLAMA_MODEL,
    audience="product managers",
    purpose="evaluating RAG vendors",
)

print("\n📋 Manager questions:")
for _, row in mgr_result.df.iterrows():
    print(f"  Q: {row['question'][:80]}")

## 9. Compare results

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Scenario": ["Baseline", "Detailed", "Developers", "Managers"],
    "Accuracy": [
        f"{result.accuracy:.0%}",
        f"{detailed_result.accuracy:.0%}",
        f"{dev_result.accuracy:.0%}",
        f"{mgr_result.accuracy:.0%}",
    ],
    "Avg Score": [
        f"{result.avg_score:.1f}/5",
        f"{detailed_result.avg_score:.1f}/5",
        f"{dev_result.avg_score:.1f}/5",
        f"{mgr_result.avg_score:.1f}/5",
    ],
    "Sample Question": [
        result.df.iloc[0]["question"][:60] + "..." if len(result.df) > 0 else "N/A",
        detailed_result.df.iloc[0]["question"][:60] + "..." if len(detailed_result.df) > 0 else "N/A",
        dev_result.df.iloc[0]["question"][:60] + "..." if len(dev_result.df) > 0 else "N/A",
        mgr_result.df.iloc[0]["question"][:60] + "..." if len(mgr_result.df) > 0 else "N/A",
    ],
})
display(comparison)

## 10. CLI equivalent

You can do the same from the command line:

In [ ]:
# Generate QA pairs with Ollama
!ragscore generate sample_doc.txt -p ollama -m {OLLAMA_MODEL}

# View generated QAs
!head -3 output/generated_qas.jsonl | python -m json.tool

---

## 📚 Resources

- **GitHub**: https://github.com/HZYAI/RagScore
- **PyPI**: https://pypi.org/project/ragscore/
- **Ollama**: https://ollama.ai
- **Website**: https://ragscore.io

⭐ If RAGScore helps you, give us a star on GitHub!